# Libreria Online: Scraping Con Paginacion

![Libros](https://images.unsplash.com/photo-1512820790803-83ca734da794?auto=format&fit=crop&w=1200&q=60)

El dataset protagonista son libros de una tienda online de practica.
En este notebook recorreras varias paginas y unificaras todo en un solo DataFrame.

# Ejercicio 2 (Guiado): Paginacion

Objetivo: recorrer varias paginas, consolidar los datos y visualizar precios por pagina.

In [ ]:
import time
import requests
from bs4 import BeautifulSoup
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')

## Paso 1: Configuracion

In [ ]:
BASE_URL = 'https://books.toscrape.com/catalogue/page-{}.html'
MAX_PAGINAS = 3
registros = []

## Paso 2: Loop de scraping por paginas

In [ ]:
for pagina in range(1, MAX_PAGINAS + 1):
    url = BASE_URL.format(pagina)
    print(f'Procesando pagina {pagina}: {url}')

    response = requests.get(url, timeout=20)
    response.raise_for_status()

    soup = BeautifulSoup(response.text, 'html.parser')
    libros = soup.select('article.product_pod')

    for libro in libros:
        titulo = libro.h3.a.get('title', '').strip()
        precio_raw = libro.select_one('p.price_color').get_text(strip=True)
        stock = libro.select_one('p.instock.availability').get_text(' ', strip=True)

        registros.append({
            'pagina': pagina,
            'titulo': titulo,
            'precio_raw': precio_raw,
            'stock': stock
        })

    time.sleep(1)

print('Total registros:', len(registros))

## Paso 3: Limpieza y DataFrame

In [ ]:
df = pd.DataFrame(registros)
df['precio'] = (
    df['precio_raw']
    .str.replace('£', '', regex=False)
    .str.replace('Â', '', regex=False)
    .astype(float)
)

df.head()

## Paso 4: Visualización exploratoria del dataset

1) Distribucion de precios
2) Precio medio por pagina

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(df['precio'], kde=True, ax=axes[0], color='#457b9d')
axes[0].set_title('Distribucion de precios')

prom_pagina = df.groupby('pagina', as_index=False)['precio'].mean()
sns.barplot(data=prom_pagina, x='pagina', y='precio', ax=axes[1], color='#e76f51')
axes[1].set_title('Precio medio por pagina')

plt.tight_layout()
plt.show()

In [ ]:
df.to_csv('libros_paginacion.csv', index=False, encoding='utf-8')
print('Archivo generado: libros_paginacion.csv')

## Checklist
- [ ] He recorrido varias paginas
- [ ] He unificado los datos en un solo DataFrame
- [ ] He limpiado precios
- [ ] He generado 2 visualizaciones
- [ ] He exportado el CSV final